# Per-Category Evaluation — All 15 MVTec AD Categories

**Purpose**: Evaluate the HOG+SVM pipeline on all 15 MVTec AD categories to demonstrate that the pipeline generalizes beyond `metal_nut`.

**Evaluation protocol**: follows Bergmann et al. (2021) — one model per category, metrics reported per category and aggregated as mean.

**Why per-category and not a single multi-category model?**  
Each category has a distinct visual domain (textures vs. rigid objects, different defect types). A single model would conflate these distributions, making 'good' ambiguous across categories. Per-category training ensures each model learns a coherent decision boundary.

**Note**: EfficientNet per-category would take ~60 minutes (15 × 4 min). HOG+SVM is evaluated here; EfficientNet on `metal_nut` (Commit 6) serves as the deep learning reference point.

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.dataset import MVTEC_CATEGORIES, MVTecTorchDataset
from src.preprocessing import preprocess
from src.features import extract_hog
from src.models.classical import ClassicalClassifier
from src.evaluate import evaluate_classification, results_row

DATA_ROOT    = Path('../data/mvtec_ad')
RANDOM_STATE = 42

print(f'Categories to evaluate: {len(MVTEC_CATEGORIES)}')
print(MVTEC_CATEGORIES)

## 1. Run HOG+SVM on all 15 categories

For each category: collect paths → stratified 70/30 split → HOG extraction → SVM training → evaluation.

In [ ]:
all_results = []
failed      = []

for category in MVTEC_CATEGORIES:
    cat_path = DATA_ROOT / category
    if not cat_path.exists():
        print(f'  [SKIP] {category} — not found')
        failed.append(category)
        continue

    try:
        paths, labels = MVTecTorchDataset.collect_paths(cat_path)
        n_good    = labels.count(0)
        n_defect  = labels.count(1)

        if n_defect == 0:
            print(f'  [SKIP] {category} — no defective samples')
            failed.append(category)
            continue

        train_p, test_p, train_l, test_l = train_test_split(
            paths, labels, test_size=0.30,
            random_state=RANDOM_STATE, stratify=labels
        )

        X_train = np.array([extract_hog(preprocess(p)) for p in train_p])
        X_test  = np.array([extract_hog(preprocess(p)) for p in test_p])
        y_train = np.array(train_l)
        y_test  = np.array(test_l)

        clf    = ClassicalClassifier().fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        metrics = evaluate_classification(y_test, y_pred, model_name=category)
        row     = results_row(metrics)
        row['n_total']  = len(paths)
        row['n_good']   = n_good
        row['n_defect'] = n_defect
        all_results.append(row)

        print(f'  {category:15s}: F1={metrics["f1_binary"]:.3f}  Recall={metrics["recall_defect"]:.3f}  '
              f'({n_good}g / {n_defect}d)')

    except Exception as e:
        print(f'  [ERROR] {category}: {e}')
        failed.append(category)

print(f'\nCompleted: {len(all_results)}/15 categories')
if failed:
    print(f'Skipped: {failed}')

## 2. Results table

In [ ]:
df = pd.DataFrame(all_results).rename(columns={'Model': 'Category'})
df = df.sort_values('F1 (defect)', ascending=False).reset_index(drop=True)

# Add mean row
mean_row = df[['Accuracy','F1 (defect)','Recall (defect)','Precision (defect)']].mean()
mean_row['Category'] = 'MEAN'
df_display = pd.concat([df[['Category','Accuracy','F1 (defect)','Recall (defect)','Precision (defect)']],
                         mean_row.to_frame().T], ignore_index=True)

print('=== HOG+SVM Per-Category Results ===')
print(df_display.to_string(index=False))
print(f'\nMean F1 (defect): {df["F1 (defect)"].mean():.4f}')
print(f'Mean Recall:      {df["Recall (defect)"].mean():.4f}')
print(f'Best category:    {df.iloc[0]["Category"]} (F1={df.iloc[0]["F1 (defect)"]:.3f})')
print(f'Worst category:   {df.iloc[-1]["Category"]} (F1={df.iloc[-1]["F1 (defect)"]:.3f})')

## 3. Bar chart — F1 per category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

categories = df['Category'].tolist()
mean_f1    = df['F1 (defect)'].mean()

# F1 bar chart
colors = ['tomato' if f < mean_f1 else 'steelblue' for f in df['F1 (defect)']]
axes[0].barh(categories[::-1], df['F1 (defect)'][::-1], color=colors[::-1])
axes[0].axvline(mean_f1, color='black', linestyle='--', lw=1.5, label=f'Mean={mean_f1:.3f}')
axes[0].set_xlabel('F1 (defect class)')
axes[0].set_title('HOG+SVM — F1 per Category')
axes[0].legend(); axes[0].set_xlim(0, 1)

# Recall bar chart
mean_rec = df['Recall (defect)'].mean()
colors_r = ['tomato' if r < mean_rec else 'steelblue' for r in df['Recall (defect)']]
axes[1].barh(categories[::-1], df['Recall (defect)'][::-1], color=colors_r[::-1])
axes[1].axvline(mean_rec, color='black', linestyle='--', lw=1.5, label=f'Mean={mean_rec:.3f}')
axes[1].set_xlabel('Recall (defect class)')
axes[1].set_title('HOG+SVM — Recall per Category')
axes[1].legend(); axes[1].set_xlim(0, 1)

plt.suptitle('Per-Category Evaluation — All 15 MVTec AD Categories\n(HOG+SVM, stratified 70/30 split)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Analysis — why do some categories perform better?

HOG captures gradient structure. Categories where defects produce strong, localized gradient changes (cracks, holes) tend to be easier than categories where defects are subtle color or texture variations.

In [ ]:
# Reference: metal_nut EfficientNet result for comparison
efficientnet_metal_nut = {
    'Category': 'metal_nut (EfficientNet t=0.3)',
    'Accuracy': 0.8911, 'F1 (defect)': 0.7843,
    'Recall (defect)': 0.7143, 'Precision (defect)': 0.8696
}

metal_nut_hog = df[df['Category'] == 'metal_nut'].iloc[0]

print('=== metal_nut — HOG+SVM vs EfficientNet ===')
comp = pd.DataFrame([
    {'Model': 'HOG+SVM',
     'F1': metal_nut_hog['F1 (defect)'],
     'Recall': metal_nut_hog['Recall (defect)'],
     'Precision': metal_nut_hog['Precision (defect)']},
    {'Model': 'EfficientNet (t=0.3)',
     'F1': 0.7843, 'Recall': 0.7143, 'Precision': 0.8696},
])
print(comp.to_string(index=False))
print()
print('=== Dataset statistics per category ===')
stats = df[['Category','n_total','n_good','n_defect']].copy()
stats['defect_ratio'] = (stats['n_defect'] / stats['n_total']).round(3)
print(stats.sort_values('defect_ratio').to_string(index=False))

## Summary

| Metric | Value |
|---|---|
| Categories evaluated | 15 |
| Mean F1 (defect) | *see table* |
| Mean Recall (defect) | *see table* |
| Best category | *see chart* |
| Worst category | *see chart* |

**Key finding**: HOG+SVM performance varies significantly across categories. Categories with structural defects (cracks, holes) that produce strong gradient changes score higher than those with subtle texture or color defects. This variation motivates the use of learned features (EfficientNet) for challenging categories.

**Protocol reference**: Bergmann et al., *The MVTec Anomaly Detection Dataset*, IJCV 2021.